# Anti-SCAM Mobile Money Togo - AI4Youth 2026
## Catégorie: Data Science | Dev Python

**Problème:** faux SMS Flooz/TMoney au Togo (`vous avez reçu...`, `compte bloqué...`). Pertes financières, surtout personnes âgées et revendeuses.

**Solution:** classifieur SMS SCAM/HAM FR + argot local. TF-IDF + LogisticRegression vs XGBoost.

**Datasets:** UCI SMS Spam (5169 après dedup) + 30 exemples Togo x10 oversampling = 5469 total. Reproductible.


In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
from xgboost import XGBClassifier
import matplotlib.pyplot as plt, seaborn as sns, joblib

BASE = Path('.')
DATA = BASE / 'data'
uci = pd.read_csv(DATA / 'SMSSpamCollection', sep='\t', header=None, names=['label','text'])
uci['label'] = uci['label'].map({'ham':0,'spam':1})
togo = pd.read_csv(DATA / 'togo_examples.csv')
togo['label'] = togo['label'].map({'ham':0,'spam':1})
togo_aug = pd.concat([togo]*10, ignore_index=True)  # oversampling tokens FR locaux
uci_dedup = uci.drop_duplicates()
df = pd.concat([uci_dedup[['label','text']], togo_aug[['label','text']]], ignore_index=True)
df['longueur'] = df['text'].str.len()
print(f'Total {len(df)} | spam {df.label.sum()} ({df.label.mean()*100:.1f}%)')
print(df.groupby('label')['longueur'].describe())
print('Doublons UCI d origine:', uci.duplicated().sum(), '| Manquants:', df.isnull().sum().sum())


## 1. EDA - Analyse exploratoire (exigée PDF)
- Histogramme longueurs, corrélation, déséquilibre classes, synthèse.


In [ ]:
sns.histplot(data=df, x='longueur', hue='label', bins=50)
plt.title('Longueur SMS ham(0) vs spam(1) - spam plus longs ~130 vs 70')
plt.savefig('eda_longueur.png'); plt.show()

print(df['label'].value_counts(normalize=True))
# Synthèse: 15.3% spam -> déséquilibre modéré, stratify obligatoire. Spam plus longs, vocabulaire promo/urgence.


## 2. Preprocessing + Split train/val/test
- Nettoyage: dedup UCI fait, imputation non nécessaire (0 manquants).
- TF-IDF 1-2 grams, 8000 features. Split 70/15/15 stratifié.


In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(df['text'], df['label'], test_size=0.15, random_state=42, stratify=df['label'])
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp)
print(f'Train {len(X_train)} / Val {len(X_val)} / Test {len(X_test)}')
vec = TfidfVectorizer(lowercase=True, ngram_range=(1,2), max_features=8000)
Xtr = vec.fit_transform(X_train); Xv = vec.transform(X_val); Xte = vec.transform(X_test)
print('Vocab:', len(vec.vocabulary_), '| Ex tokens Flooz:', 'flooz' in vec.vocabulary_)


## 3. Modélisation - 2 approches comparées


In [ ]:
lr = LogisticRegression(max_iter=1000).fit(Xtr, y_train)
print('=== LogReg VAL ===')
print(classification_report(y_val, lr.predict(Xv), target_names=['ham','spam']))

xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.9, colsample_bytree=0.9, eval_metric='logloss', random_state=42)
xgb.fit(Xtr, y_train)
print('=== XGBoost VAL ===')
print(classification_report(y_val, xgb.predict(Xv), target_names=['ham','spam']))

f1_lr = f1_score(y_val, lr.predict(Xv), pos_label=1)
f1_xgb = f1_score(y_val, xgb.predict(Xv), pos_label=1)
best_name, best = ('XGBoost', xgb) if f1_xgb>=f1_lr else ('LogReg', lr)
print(f'Meilleur: {best_name} F1-spam VAL {max(f1_lr,f1_xgb):.3f}')


## 4. Évaluation finale TEST + limites


In [ ]:
pt = best.predict(Xte)
print(classification_report(y_test, pt, target_names=['ham','spam']))
print('ROC-AUC:', roc_auc_score(y_test, best.predict_proba(Xte)[:,1]))
sns.heatmap(confusion_matrix(y_test, pt), annot=True, fmt='d'); plt.title(f'Confusion TEST {best_name}'); plt.savefig('confusion_test.png'); plt.show()

tdf = pd.DataFrame({'text':X_test.values,'y':y_test.values,'pred':pt})
print('Succès SCAM:', tdf[(tdf.y==1)&(tdf.pred==1)].head(2)['text'].tolist())
print('Échecs FN:', tdf[(tdf.y==1)&(tdf.pred==0)].head(2)['text'].tolist())
print('Limites: dataset EN biaisé, 30 exemples Togo insuffisants, pas de Ewe oral, à collecter 500 SMS terrain pendant sprint.')
print('Pistes: BERT multilingue, Whisper vocal, app Streamlit.')
joblib.dump(vec,'tfidf.joblib'); joblib.dump(best,'model.joblib')


## 5. Démo Togo


In [ ]:
for txt in ['Vous avez recu 50000 FCFA via Flooz. Envoyez votre code PIN pour debloquer', 'Salut, on se voit au marche Hedzranawoe demain?']:
    p = best.predict_proba(vec.transform([txt]))[0,1]
    print(f"[{'SCAM' if p>0.5 else 'HAM'} {p:.2f}] {txt}")
